# Collect ~1000 Applied CS Papers from arXiv

This notebook queries arXiv for applied CS categories, applies a lightweight applied-keyword filter,
removes duplicates, and saves outputs to CSV and JSON.

Target categories: `cs.LG` (Machine Learning), `cs.CV` (Computer Vision), `cs.CL` (Computation & Language),
`cs.RO` (Robotics), `cs.SE` (Software Engineering), `cs.DB` (Databases), `cs.AR` (Hardware Architecture),
`cs.NI` (Networking & Internet), `cs.HC` (Human-Computer Interaction), `cs.IR` (Information Retrieval),
`cs.AI` (Artificial Intelligence).

Rate-limit rules followed: 100 results per request, 3-second delay between requests.

In [ ]:
# %pip install requests pandas
import re
import time
from pathlib import Path
import xml.etree.ElementTree as ET

import pandas as pd
import requests

BASE_URL = "http://export.arxiv.org/api/query"
CATEGORIES = ["cs.LG", "cs.CV", "cs.CL", "cs.RO", "cs.SE", "cs.DB", "cs.AR", "cs.NI", "cs.HC", "cs.IR", "cs.AI"]
SEARCH_QUERY = " OR ".join(f"cat:{cat}" for cat in CATEGORIES)

TARGET_PAPERS = 1000
MAX_RESULTS_PER_CALL = 100
MAX_REQUESTS = 50
REQUEST_DELAY_SECONDS = 3
REQUEST_TIMEOUT_SECONDS = 60
MIN_KEYWORD_HITS = 2

APPLIED_KEYWORDS = [
    "experiment",
    "experiments",
    "experimental",
    "benchmark",
    "benchmarks",
    "benchmarking",
    "dataset",
    "datasets",
    "evaluation",
    "evaluate",
    "evaluating",
    "empirical",
    "empirically",
    "ablation",
    "ablation study",
    "implementation",
    "implement",
    "implemented",
    "we propose",
    "we introduce",
    "we present",
    "we build",
    "we design",
    "we develop",
    "we train",
    "we fine-tune",
    "we deploy",
    "performance",
    "accuracy",
    "throughput",
    "real-world",
    "real world",
    "state-of-the-art",
    "sota",
    "outperforms",
    "outperform",
    "user study",
    "human evaluation",
    "baseline",
    "baselines",
    "training",
    "inference",
    "results show",
    "we demonstrate",
    "we evaluate",
    "our method",
    "our model",
    "our system",
    "our approach",
    "our framework",
]

THEORY_EXCLUSION_KEYWORDS = [
    "theorem",
    "lemma",
    "corollary",
    "we prove",
    "formal proof",
    "np-hard",
    "np-complete",
    "lower bound",
    "upper bound",
    "convergence proof",
    "complexity analysis",
]

OUTPUT_DIR = Path("Dataset collection")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_CSV = OUTPUT_DIR / "applied_papers_1000.csv"
OUTPUT_JSON = OUTPUT_DIR / "applied_papers_1000.json"

ATOM_NS = "{http://www.w3.org/2005/Atom}"
ARXIV_NS = "{http://arxiv.org/schemas/atom}"

session = requests.Session()
session.headers.update({
    "User-Agent": "ResearchPaperCategorizer/1.0 (mailto:replace-with-your-email@example.com)"
})

In [ ]:
def clean_text(value: str) -> str:
    return " ".join((value or "").split())


APPLIED_PATTERNS = [re.compile(rf"\b{re.escape(keyword)}\b") for keyword in APPLIED_KEYWORDS]
THEORY_EXCLUSION_PATTERNS = [re.compile(rf"\b{re.escape(keyword)}\b") for keyword in THEORY_EXCLUSION_KEYWORDS]


def keyword_hits(text: str) -> list[str]:
    lowered = text.lower()
    return [keyword for keyword, pattern in zip(APPLIED_KEYWORDS, APPLIED_PATTERNS) if pattern.search(lowered)]


def theory_exclusion_hits(text: str) -> list[str]:
    lowered = text.lower()
    return [keyword for keyword, pattern in zip(THEORY_EXCLUSION_KEYWORDS, THEORY_EXCLUSION_PATTERNS) if pattern.search(lowered)]


def parse_entry(entry) -> dict:
    title = clean_text(entry.findtext(f"{ATOM_NS}title", default=""))
    abstract = clean_text(entry.findtext(f"{ATOM_NS}summary", default=""))
    arxiv_url = clean_text(entry.findtext(f"{ATOM_NS}id", default=""))
    published = clean_text(entry.findtext(f"{ATOM_NS}published", default=""))

    cat_terms = []
    for cat in entry.findall(f"{ATOM_NS}category"):
        term = clean_text(cat.attrib.get("term", ""))
        if term:
            cat_terms.append(term)

    primary_node = entry.find(f"{ARXIV_NS}primary_category")
    primary_category = ""
    if primary_node is not None:
        primary_category = clean_text(primary_node.attrib.get("term", ""))
    if not primary_category and cat_terms:
        primary_category = cat_terms[0]

    arxiv_id = arxiv_url.rsplit("/", 1)[-1] if arxiv_url else ""

    return {
        "arxiv_id": arxiv_id,
        "title": title,
        "abstract": abstract,
        "published": published,
        "primary_category": primary_category,
        "categories": " ".join(sorted(set(cat_terms))),
        "arxiv_url": arxiv_url,
    }


def fetch_papers(start: int, max_results: int = 100) -> list[dict]:
    params = {
        "search_query": SEARCH_QUERY,
        "start": start,
        "max_results": max_results,
        "sortBy": "submittedDate",
        "sortOrder": "descending",
    }
    response = session.get(BASE_URL, params=params, timeout=REQUEST_TIMEOUT_SECONDS)
    response.raise_for_status()

    root = ET.fromstring(response.text)
    entries = root.findall(f"{ATOM_NS}entry")
    return [parse_entry(entry) for entry in entries]


def is_applied_like(paper: dict, min_hits: int = 2) -> bool:
    text = f"{paper['title']} {paper['abstract']}"
    hits = keyword_hits(text)
    paper["keyword_hits"] = ", ".join(hits)
    paper["keyword_hit_count"] = len(hits)
    return len(hits) >= min_hits


def passes_theory_exclusion(paper: dict) -> bool:
    """Return True if the paper does NOT trigger >= 3 theory exclusion keywords."""
    text = f"{paper['title']} {paper['abstract']}"
    excl_hits = theory_exclusion_hits(text)
    paper["theory_exclusion_hits"] = ", ".join(excl_hits)
    paper["theory_exclusion_hit_count"] = len(excl_hits)
    return len(excl_hits) < 3

In [ ]:
applied_papers = []
seen_ids = set()
seen_titles = set()
rejected_by_exclusion = 0

start = 0
for request_idx in range(1, MAX_REQUESTS + 1):
    batch = fetch_papers(start=start, max_results=MAX_RESULTS_PER_CALL)
    if not batch:
        print("No more results returned by arXiv.")
        break

    accepted = 0
    for paper in batch:
        if not paper["title"] or not paper["abstract"]:
            continue

        title_key = re.sub(r"\s+", " ", paper["title"].lower()).strip()
        if paper["arxiv_id"] in seen_ids or title_key in seen_titles:
            continue

        if is_applied_like(paper, min_hits=MIN_KEYWORD_HITS):
            if passes_theory_exclusion(paper):
                seen_ids.add(paper["arxiv_id"])
                seen_titles.add(title_key)
                applied_papers.append(paper)
                accepted += 1
            else:
                rejected_by_exclusion += 1

        if len(applied_papers) >= TARGET_PAPERS:
            break

    print(
        f"Request {request_idx}/{MAX_REQUESTS} | start={start} | fetched={len(batch)} | accepted={accepted} | total={len(applied_papers)} | excl_rejected={rejected_by_exclusion}"
    )

    if len(applied_papers) >= TARGET_PAPERS:
        break

    start += MAX_RESULTS_PER_CALL
    time.sleep(REQUEST_DELAY_SECONDS)

print(f"Collected {len(applied_papers)} applied-like papers.")
print(f"Rejected by theory exclusion filter: {rejected_by_exclusion}")

In [ ]:
df = pd.DataFrame(applied_papers)
if df.empty:
    raise RuntimeError("No papers collected. Try reducing MIN_KEYWORD_HITS or increasing MAX_REQUESTS.")

df = df.head(TARGET_PAPERS).copy()
df.insert(0, "label", "APPLIED")
df.insert(1, "source_query", SEARCH_QUERY)

# Keep a project-friendly column order.
ordered_cols = [
    "label",
    "title",
    "abstract",
    "primary_category",
    "categories",
    "keyword_hit_count",
    "keyword_hits",
    "theory_exclusion_hit_count",
    "theory_exclusion_hits",
    "published",
    "arxiv_id",
    "arxiv_url",
    "source_query",
]
df = df[ordered_cols]

df.to_csv(OUTPUT_CSV, index=False)
df.to_json(OUTPUT_JSON, orient="records", indent=2, force_ascii=False)

print(f"Saved {len(df):,} rows to {OUTPUT_CSV}")
print(f"Saved {len(df):,} rows to {OUTPUT_JSON}")
print()
print("Top primary categories:")
print(df["primary_category"].value_counts().head(10))
print()
print("Keyword hit distribution:")
print(df["keyword_hit_count"].value_counts().sort_index())

df.head(5)

In [ ]:
# Quick manual spot-check (10 examples)
preview_cols = ["title", "primary_category", "keyword_hits"]
df.sample(n=min(10, len(df)), random_state=42)[preview_cols].reset_index(drop=True)

In [ ]:
# ── Category Distribution & Quality Summary ─────────────────────────────────
print(f"{'='*60}")
print(f"APPLIED DATASET QUALITY SUMMARY")
print(f"{'='*60}")
print(f"Total papers collected  : {len(df):,}")
print(f"Rejected by theory filter: {rejected_by_exclusion:,}")
print()

print("Papers per arXiv category (primary):")
print(df["primary_category"].value_counts().to_string())
print()

# Top 10 most frequent individual applied keywords across all abstracts
from collections import Counter

all_hits = []
for hits_str in df["keyword_hits"]:
    if hits_str:
        all_hits.extend([k.strip() for k in hits_str.split(",") if k.strip()])

keyword_counter = Counter(all_hits)
print("Top 10 most frequent applied keywords:")
for kw, count in keyword_counter.most_common(10):
    print(f"  {kw:<30} {count:>5}")
print()

print(f"Rejected by theory exclusion filter: {rejected_by_exclusion:,}")
print(f"{'='*60}")